# Hafta 2 — Colab Duman Testi (smoke test)

**Amac:** Gece boyu surecek uretimi baslatmadan ONCE, 10 dakikada su sorulari cevaplamak:

1. Hangi modeller bu GPU'ya siğiyor?
2. Depolar hala erisilebilir mi? (RunwayML SD1.5 depolarini kaldirdi)
3. Goruntu basina kac saniye? Yani 300 goruntu kac saat?
4. Inpainting hatti (M1/M2) gercekten calisiyor mu?
5. Prompt disiplini tutuyor mu — ciktilar "amator telefon fotografi" gibi mi, yoksa stok fotograf gibi mi?

**Bu notebook hicbir sey uretmez, sadece OLCER.** Sonunda bir karar tablosu basar.

> Colab'da: `Runtime -> Change runtime type -> T4 GPU` sectiginden emin ol.

## 0. Ortam

In [ ]:
import subprocess, sys, torch, platform

print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name}")
    print(f"VRAM   : {p.total_memory/1e9:.1f} GB")
    print(f"Compute: {p.major}.{p.minor}  (bf16 destegi: {'VAR' if p.major >= 8 else 'YOK -> fp16 kullanilacak'})")
else:
    print("\n>>> GPU YOK. Runtime -> Change runtime type -> T4 GPU sec ve yeniden calistir.")

In [ ]:
# Bagimliliklar. Colab'da torch zaten kurulu; sadece eksikleri ekliyoruz.
!pip install -q diffusers accelerate safetensors transformers sentencepiece protobuf
import diffusers
print("diffusers:", diffusers.__version__)

## 1. Repo

GitHub adresini kendi reponla degistir. Veri (`data/`) repoda DEGIL — bu test
veri gerektirmiyor, sentetik uretimi kendi girdisini kendisi olusturuyor.

In [ ]:
REPO_URL = "https://github.com/Tunahan-46/insurance-image-forensics.git"

import os
from pathlib import Path

if not Path("insurance-image-forensics").exists():
    !git clone -q $REPO_URL
os.chdir("/content/insurance-image-forensics")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

from src.data.generators.pipelines import MODEL_REGISTRY
print("\nKayitli modeller:")
for k, s in MODEL_REGISTRY.items():
    print(f"  {k:<14} ~{s.approx_gb:>4.1f} GB   test_only={s.test_only}")

## 2. Prompt disiplini (GPU gerektirmez)

Plan 4.3'un altin kurali: prompt'a "8k, cinematic, professional" sizarsa
model studyo kalitesinde gorsel uretir, dedektorun isi trivial hale gelir ve
**%99 accuracy'nin sebebi model degil prompt olur.**

In [ ]:
from src.data.generators.prompts import build_prompt_batch, combination_space, NEGATIVE_PROMPT

print(f"Teorik kombinasyon: {combination_space():,}\n")
for s in build_prompt_batch(4, seed=42):
    print(f"[{'HASARLI' if s.has_damage else 'HASARSIZ'}] {s.positive}\n")
print("negative:", NEGATIVE_PROMPT)

## 3. Model testleri

Her model AYRI hucrede. Biri patlarsa digerleri etkilenmesin diye.
Her testten sonra VRAM temizleniyor.

In [ ]:
import gc, time, torch
from pathlib import Path
from src.data.generators.fully_synthetic import generate as gen_synth

RESULTS = {}   # model -> {"ok":bool, "sn_per_img":float, "vram_gb":float, "hata":str}
OUT = Path("/content/smoke_out")

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def smoke(model, n=2):
    free()
    t0 = time.time()
    try:
        res = gen_synth(model, n, out_root=OUT, seed=1, device="cuda", resume=False)
        el = time.time() - t0
        vram = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0
        RESULTS[model] = {"ok": len(res) > 0, "sn_per_img": el/max(1,len(res)),
                          "vram_gb": vram, "hata": ""}
        print(f"\n[OK] {model}: {len(res)} goruntu, {el/max(1,len(res)):.1f} sn/goruntu, "
              f"tepe VRAM {vram:.1f} GB")
    except Exception as e:
        RESULTS[model] = {"ok": False, "sn_per_img": 0, "vram_gb": 0,
                          "hata": f"{type(e).__name__}: {str(e)[:300]}"}
        print(f"\n[BASARISIZ] {model}\n  {type(e).__name__}: {str(e)[:300]}")
    finally:
        free()

### 3a. SD 1.5 — ~4 GB, en guvenli

In [ ]:
smoke('sd15', n=2)

### 3b. SDXL — ~7 GB, ana uretici

In [ ]:
smoke('sdxl', n=2)

### 3c. FLUX.1-schnell — ~24 GB, RISKLI

12 milyar parametre. Ucretsiz T4'te (16 GB, bf16 yok) buyuk ihtimalle
yetmeyecek ya da cok yavas olacak. **Basarisiz olmasi bir felaket degil** --
"gorulmemis uretici" deneyini SD1.5-train / SDXL-test kurgusuyla da yapabiliriz.

Bu hucre uzun surebilir (indirme). Sabirsizlaninca durdurup gecebilirsin.

In [ ]:
smoke('flux_schnell', n=1)

### 3d. sd_turbo — flux_schnell yerine, ~5 GB

flux_schnell RAM tasirdigi icin devre disi. sd_turbo halka acik (HF token gerektirmez), SD 2.1'in tek-adimda calisan damitilmis surumu -- pipelines.py'ye eklendi.

In [ ]:
smoke('sd_turbo', n=2)

## 4. Inpainting hatti (M1) — projenin en kritik senaryosu

Gercek CarDD verisi Colab'da yok. Ama bu test **veri kalitesini degil,
HATTIN MEKANIGINI** olcuyor: VRAM yetiyor mu, maske dogru geciyor mu,
kabul kapisi calisiyor mu.

Girdi olarak az once uretilen sentetik goruntuleri "gercek" yerine koyuyoruz.

In [ ]:
# KURAL 1: cv2.imread/imwrite DOGRUDAN cagrilmaz -- src.data.imageio kullanilir.
# Colab yolu ASCII oldugu icin burada fark etmez, ama kurali notebook'ta
# bozmak, kopyala-yapistir yoluyla Windows tarafina sizmasinin en kisa yolu.
import cv2, numpy as np
from src.data.imageio import imread, imwrite
from src.data.manifest import new_manifest, add_row, save_manifest

srcs = sorted(OUT.rglob("*.png"))
print(f"Girdi olarak kullanilacak goruntu: {len(srcs)}")

if not srcs:
    print(">>> Once 3a veya 3b calismali.")
else:
    df = new_manifest()
    mdir = Path("/content/smoke_masks"); mdir.mkdir(exist_ok=True)
    for i, p in enumerate(srcs[:4]):
        im = imread(p); h, w = im.shape[:2]
        # Sahte 'hasar maskesi' -- gercek CarDD maskesinin yerine gecer
        m = np.zeros((h, w), np.uint8)
        cv2.circle(m, (int(w*0.4), int(h*0.6)), int(min(h, w)*0.08), 255, -1)
        mp = mdir / f"{p.stem}_dmg.png"; imwrite(mp, m)
        df = add_row(df, source_image_id=f"smoke_{i:03d}", path=str(p), label="real",
                     width=w, height=h, split="train",
                     gen_params={"damage_mask_path": str(mp)})
    save_manifest(df, "/content/smoke_manifest.parquet")

In [ ]:
from src.data.generators.inpaint_add import generate as gen_inpaint

free()
t0 = time.time()
try:
    res = gen_inpaint("/content/smoke_manifest.parquet", 2, model="sdxl",
                      out_root="/content/smoke_m1", seed=5, device="cuda", resume=False)
    el = time.time() - t0
    vram = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0
    RESULTS["inpaint_sdxl"] = {"ok": len(res) > 0, "sn_per_img": el/max(1,len(res)),
                               "vram_gb": vram, "hata": ""}
    print(f"\n[OK] inpaint/sdxl: {len(res)} ornek, {el/max(1,len(res)):.1f} sn/goruntu, "
          f"tepe VRAM {vram:.1f} GB")
    for r in res:
        print(f"  {r.manip_type}  maske alani "
              f"{r.gen_params['mask_area_frac']*100:.1f}%  "
              f"degisen {r.gen_params['changed_frac_in_mask']*100:.0f}%  "
              f"maske disi sizinti {r.gen_params['leak_frac_outside_mask']*100:.2f}%")
except Exception as e:
    RESULTS["inpaint_sdxl"] = {"ok": False, "sn_per_img": 0, "vram_gb": 0,
                               "hata": f"{type(e).__name__}: {str(e)[:300]}"}
    print(f"\n[BASARISIZ] inpaint/sdxl\n  {type(e).__name__}: {str(e)[:300]}")
finally:
    free()

## 5. Ciktilari GOZLE incele

Plan 4.3: "Urettigin goruntulerden rastgele 100'unu **kendin gozle incele.**"
Burada 4-6 tane bakiyoruz ama kritik soru ayni:

**Bunlar amator telefon fotografi gibi mi duruyor, yoksa stok fotograf gibi mi?**

Stok fotograf gibiyse prompt disiplini tutmamis demektir ve dedektorun
alacagi yuksek AUC anlamsiz olur.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

imgs = sorted(OUT.rglob("*.png")) + sorted(Path("/content/smoke_m1").glob("*.png"))
imgs = imgs[:6]
if imgs:
    fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, imgs):
        ax.imshow(Image.open(p)); ax.axis("off")
        ax.set_title(p.parent.name, fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("Gosterilecek goruntu yok.")

## 6. KARAR TABLOSU

In [ ]:
HEDEF = {"sd15": 150, "sdxl": 150, "sd_turbo": 100, "inpaint_sdxl": 400}

print(f"{'bilesen':<16}{'durum':<10}{'sn/goruntu':<13}{'VRAM':<10}{'hedef':<8}{'tahmini sure'}")
print("-"*76)
toplam = 0.0
for k, v in RESULTS.items():
    n = HEDEF.get(k, 0)
    if v["ok"]:
        saat = v["sn_per_img"] * n / 3600
        toplam += saat
        print(f"{k:<16}{'CALISTI':<10}{v['sn_per_img']:<13.1f}{v['vram_gb']:<10.1f}"
              f"{n:<8}{saat:.1f} saat")
    else:
        print(f"{k:<16}{'PATLADI':<10}{'-':<13}{'-':<10}{n:<8}-")

print("-"*76)
print(f"{'TOPLAM':<16}{'':<10}{'':<13}{'':<10}{'':<8}{toplam:.1f} saat")

basarisiz = [k for k, v in RESULTS.items() if not v["ok"]]
print()
if basarisiz:
    print(f">>> Calismayanlar: {basarisiz}")
    for k in basarisiz:
        print(f"    {k}: {RESULTS[k]['hata']}")
    print()
    if "flux_schnell" in basarisiz and len(basarisiz) == 1:
        print(">>> SADECE FLUX patladi -- bu SORUN DEGIL.")
        print(">>> 'Gorulmemis uretici' deneyini su kurguyla yap:")
        print(">>>   train/val -> SD 1.5")
        print(">>>   test      -> SDXL  (model bunu hic gormez)")
        print(">>> pipelines.py icinde sdxl'in test_only degerini True yapman yeterli.")
else:
    print(">>> Hepsi calisti. Gece boyu uretim guvenli.")

if toplam > 8:
    print(f"\n>>> UYARI: {toplam:.1f} saat, ucretsiz Colab oturum limitini asar.")
    print(">>> Hedefleri dusur ya da uretimi birkac oturuma bol.")
    print(">>> Not: ureticiler resume=True ile calisir, kopan oturumdan devam eder.")

---

## 7. Gece boyu uretim

Duman testi gectiyse buradan devam. Hucreleri SIRAYLA calistir.

**Onemli:** S katmani (tam sentetik) veri GEREKTIRMEZ, hemen baslatilabilir.
M1/M2 icin CarDD'nin Drive'da olmasi gerekir -- asagidaki hucre onu bagliyor.

In [ ]:
# 7a. Drive'i bagla ve CarDD'yi YEREL DISKE ac.
#
# NEDEN symlink degil de acma: Colab'in Drive baglantisi FUSE uzerinden
# calisir ve binlerce kucuk dosyayi oradan okumak cok yavastir. Inpainting
# 250 goruntuyu tek tek acacak; bunlari yerel diskte tutmak uretim suresini
# ciddi olcude kisaltir. Yerel disk oturumla birlikte silinir ama zaten
# kaynak veri Drive'da duruyor.
#
# Drive'da beklenen dosyalar (yerelde `Compress-Archive` ile uretildi):
#     MyDrive/insurance-data/cardd_coco.zip        (~3 GB, goruntular)
#     MyDrive/insurance-data/cardd_sod_masks.zip   (~25 MB, hasar maskeleri)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/insurance-data')

for z in ['cardd_coco.zip', 'cardd_sod_masks.zip']:
    print(f"{z:<26}", 'VAR' if (DRIVE / z).exists() else '>>> YOK, once Drive a yukle')


In [ ]:
# 7a-2. Zipleri yerine ac. Birkac dakika surer, bir kez calistirilir.
import zipfile, time
from pathlib import Path

CARDD = Path('data/raw/cardd')
CARDD.mkdir(parents=True, exist_ok=True)

# COCO: zip icindeki kok klasor zaten CarDD_COCO
if not (CARDD / 'CarDD_COCO').exists():
    t0 = time.time()
    with zipfile.ZipFile(DRIVE / 'cardd_coco.zip') as z:
        z.extractall(CARDD)
    print(f"CarDD_COCO acildi ({time.time()-t0:.0f} sn)")

# SOD maskeleri: zip duz yapida (CarDD-TR-Mask, CarDD-VAL-Mask, CarDD-TE-Mask).
# build_manifest_v2.py bunlari CarDD_SOD/CarDD-XX/CarDD-XX-Mask altinda arar.
SOD = CARDD / 'CarDD_SOD'
if not SOD.exists():
    with zipfile.ZipFile(DRIVE / 'cardd_sod_masks.zip') as z:
        z.extractall('/content/_sodtmp')
    for split in ['TR', 'VAL', 'TE']:
        dst = SOD / f'CarDD-{split}'
        dst.mkdir(parents=True, exist_ok=True)
        src = next(Path('/content/_sodtmp').rglob(f'CarDD-{split}-Mask'))
        src.rename(dst / f'CarDD-{split}-Mask')
    print("SOD maskeleri yerlestirildi")

for p in ['CarDD_COCO/train2017', 'CarDD_COCO/val2017', 'CarDD_COCO/test2017',
          'CarDD_SOD/CarDD-TR/CarDD-TR-Mask']:
    d = CARDD / p
    print(f"  {p:<38} {len(list(d.iterdir())) if d.exists() else 'YOK':>6}")


In [ ]:
# 7b. Uretim icin manifest kur.
#
# manifest_v2.parquet repoda DEGIL (data/processed/* gitignore'da) -- ve
# olmasina gerek de yok: split'ler deterministik. CarDD'nin split'i klasor
# yapisindan (train2017/val2017/test2017) geliyor, hash'ten degil. Yani
# burada kurulan manifest, CarDD satirlari icin yereldekiyle AYNI split'i
# verir. Bu manifest sadece URETIM GIRDISI; gercek manifest yerelde kurulur.
#
# Kendi fotograflarin ve M3 ciktilarina burada ihtiyac yok, o yuzden
# "0 bulundu" satirlari normal.

!python scripts/build_manifest_v2.py

### 7c. S katmani — tam sentetik

Duman testindeki `sn/goruntu` degerine gore hedefleri ayarla. `resume=True`
varsayilan: oturum koparsa ayni hucreyi tekrar calistir, kaldigi yerden devam eder.

**flux_schnell yerine `sd_turbo` kullaniliyor.** Duman testinde flux_schnell
GPU'ya binmeden SISTEM RAM'ini tasirip oturumu cokertti (12B parametre,
~24 GB agirlik, ucretsiz Colab'a sigmiyor). `sd_turbo` (~5 GB, SD 2.1'in
tek-adimda calisan damitilmis surumu) onun yerine test-only 'gorulmemis
uretici' rolunu ustleniyor -- bkz. `src/data/generators/pipelines.py`.
sdxl'i test_only yapmak YANLIS olurdu: sdxl ayni zamanda M1/M2 inpainting
modeli, o degisiklik projenin en kritik senaryosunu (inpaint_add) durururdu.

In [ ]:
!python -m src.data.generators.fully_synthetic --model sd15 --n 150 --seed 1
!python -m src.data.generators.fully_synthetic --model sdxl --n 150 --seed 2
!python -m src.data.generators.fully_synthetic --model sd_turbo --n 100 --seed 3

### 7d. M1 / M2 — inpainting

`inpaint_add` projenin en kritik senaryosu. Reddetme orani yuksek cikarsa
(`kalite kapisinda reddedildi`) `--strength` degerini artir.

In [ ]:
!python -m src.data.generators.inpaint_add --manifest data/processed/manifest_v2.parquet \
    --n 250 --model sdxl --seed 11

!python -m src.data.generators.inpaint_remove --manifest data/processed/manifest_v2.parquet \
    --n 150 --method sd_inpaint --model sdxl --seed 12

### 7e. Ciktilari Drive'a kaydet — **BU HUCREYI ATLAMA**

Colab oturumu kapaninca `/content` altindaki her sey silinir. Gece boyu
uretim yapip bunu unutmak, sabah sifirdan baslamak demektir.

In [ ]:
import shutil, time
from pathlib import Path

OUT_ZIP = Path('/content/drive/MyDrive/insurance-data/w2_generated')
OUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

for folder in ['data/raw/synthetic', 'data/raw/manipulated']:
    p = Path(folder)
    if not p.exists():
        print(f'{folder}: yok, atlaniyor')
        continue
    n = sum(1 for _ in p.rglob('*.png'))
    name = str(OUT_ZIP) + '_' + folder.replace('/', '_')
    shutil.make_archive(name, 'zip', p)
    mb = Path(name + '.zip').stat().st_size / 1e6
    print(f'{folder}: {n} goruntu -> {name}.zip  ({mb:.0f} MB)')

print('\nSonraki adim (YERELDE):')
print('  1. zipleri indir, data/raw/ altina ac')
print('  2. python scripts/build_manifest_v2.py     # split 2854/817/381 kalmali')
print('  3. python scripts/apply_laundering.py')
print('  4. python scripts/run_e1_shortcut.py       # Gorev A artik olculebilir')